# Week 4 Day 2: Core Supervised Learning — Preprocessing, Models & Evaluation

**Scenario:** Using the Adult dataset (same split as Day 1), we build preprocessing pipelines, train two classifiers, evaluate with multiple metrics, and interpret the models.

## Task 1: Preprocessing Plan & Implementation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    ConfusionMatrixDisplay, roc_curve, precision_recall_curve
)
import joblib

# ── 1. Load Adult dataset ──────────────────────────────────────────────────────
print('Loading Adult dataset...')
adult = fetch_openml(data_id=1590, as_frame=True, parser='auto')
X = adult.data.copy()
y = (adult.target.str.strip() == '>50K').astype(int)

# Replace '?' with NaN (common encoding of missing values in this dataset)
X.replace('?', np.nan, inplace=True)

# ── 2. Same hold-out split as Day 1 (random_state=42, 20 % test) ────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print(f'Train size: {X_train.shape}, Test size: {X_test.shape}')

# ── 3. Explicitly list feature types ─────────────────────────────────────────
numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
print(f'\nNumeric features ({len(numeric_features)}):     {numeric_features}')
print(f'Categorical features ({len(categorical_features)}): {categorical_features}')

# ── 4. Preprocessing pipelines ────────────────────────────────────────────────
#
# NUMERIC — median imputation + StandardScaler
#   Why median?  Numeric columns like capital-gain / capital-loss are heavily
#   right-skewed with many zeros and a few very large values.  Median is
#   resistant to those outliers; mean would be pulled upward and distort the
#   imputed value.  KNNImputer was considered but skipped — it is O(n²) in
#   memory and too slow for ~39k training rows with 6 numeric columns.
#
# CATEGORICAL — most_frequent imputation + OneHotEncoder(handle_unknown='ignore')
#   Why OHE?  Columns like workclass, relationship, occupation are purely
#   nominal — no meaningful numeric ordering exists.  OHE avoids implying
#   false ordinal relationships that OrdinalEncoder would introduce.
#   handle_unknown='ignore' ensures unseen categories at inference time are
#   silently set to all-zero rows rather than crashing.  TargetEncoder was
#   considered but skipped because it requires extra care to avoid leakage
#   and we want a straightforward, leak-free pipeline here.

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

print('\nPreprocessor built successfully.')

## Task 2: Train Two Supervised Models (in Pipelines)

In [ ]:
# ── Logistic Regression pipeline ─────────────────────────────────────────────
# solver='lbfgs' supports L2 regularisation (default C=1.0); max_iter=1000
# avoids convergence warnings on this dataset.
logistic_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42, solver='lbfgs', max_iter=1000))
])

# ── Decision Tree pipeline ───────────────────────────────────────────────────
tree_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier(random_state=42))
])

# ⚠️  Fit on TRAINING SET ONLY — the test set is never seen during training.
print('Training Logistic Regression...')
logistic_pipeline.fit(X_train, y_train)
print('  Done.')

print('Training Decision Tree...')
tree_pipeline.fit(X_train, y_train)
print('  Done.')

## Task 3: Evaluate on Hold-Out Test (Multiple Metrics)

In [ ]:
# ── Helper: evaluate one model ────────────────────────────────────────────────
def evaluate_model(pipeline, name, X_te, y_te):
    y_pred = pipeline.predict(X_te)
    y_prob = pipeline.predict_proba(X_te)[:, 1]
    return {
        'Model'    : name,
        'Accuracy' : round(accuracy_score(y_te, y_pred), 4),
        'Precision': round(precision_score(y_te, y_pred), 4),
        'Recall'   : round(recall_score(y_te, y_pred), 4),
        'F1'       : round(f1_score(y_te, y_pred), 4),
        'ROC AUC'  : round(roc_auc_score(y_te, y_prob), 4),
        'PR AUC'   : round(average_precision_score(y_te, y_prob), 4),
        '_y_pred'  : y_pred,
        '_y_prob'  : y_prob
    }

# ── Day 1 baseline: always predict majority class (<=50K = 0) ─────────────────
y_pred_base = np.zeros(len(y_test), dtype=int)
baseline = {
    'Model'    : 'Baseline (majority-class)',
    'Accuracy' : round(accuracy_score(y_test, y_pred_base), 4),
    'Precision': 0.0,
    'Recall'   : 0.0,
    'F1'       : 0.0,
    'ROC AUC'  : 0.5,
    'PR AUC'   : round(y_test.mean(), 4),
    '_y_pred'  : y_pred_base,
    '_y_prob'  : y_pred_base.astype(float)
}

lr_res   = evaluate_model(logistic_pipeline, 'Logistic Regression', X_test, y_test)
tree_res = evaluate_model(tree_pipeline, 'Decision Tree', X_test, y_test)

# ── Comparison table ──────────────────────────────────────────────────────────
display_cols = ['Model', 'Accuracy', 'Precision', 'Recall', 'F1', 'ROC AUC', 'PR AUC']
results_df = pd.DataFrame([baseline, lr_res, tree_res])[display_cols]
print('\n=== Metric Comparison Table ===')
display(results_df)

In [ ]:
# ── ROC & Precision-Recall curves ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for res, color in [(lr_res, 'steelblue'), (tree_res, 'darkorange')]:
    name   = res['Model']
    y_prob = res['_y_prob']

    # ROC
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    axes[0].plot(fpr, tpr, color=color,
                 label=f"{name} (AUC={res['ROC AUC']:.3f})")

    # PR
    prec_vals, rec_vals, _ = precision_recall_curve(y_test, y_prob)
    axes[1].plot(rec_vals, prec_vals, color=color,
                 label=f"{name} (AUC={res['PR AUC']:.3f})")

axes[0].plot([0, 1], [0, 1], 'k--', label='Random chance')
axes[0].set(xlabel='False Positive Rate', ylabel='True Positive Rate', title='ROC Curve')
axes[0].legend()

axes[1].axhline(y=y_test.mean(), color='k', linestyle='--', label='No-skill baseline')
axes[1].set(xlabel='Recall', ylabel='Precision', title='Precision-Recall Curve')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── Confusion matrices ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ConfusionMatrixDisplay.from_estimator(
    logistic_pipeline, X_test, y_test, ax=axes[0], cmap='Blues',
    display_labels=['<=50K', '>50K']
)
axes[0].set_title('Logistic Regression — Confusion Matrix')

ConfusionMatrixDisplay.from_estimator(
    tree_pipeline, X_test, y_test, ax=axes[1], cmap='Oranges',
    display_labels=['<=50K', '>50K']
)
axes[1].set_title('Decision Tree — Confusion Matrix')
plt.tight_layout()
plt.show()

print("""
Error-type analysis:
  Both models produce more False Negatives (FN) than False Positives (FP).
  FN = predicting <=50K when the true label is >50K (missed high-earner).
  In a financial-targeting context, FNs are missed revenue opportunities
  while FPs are wasted marketing spend.  This favours optimising Recall or
  F1 over raw Accuracy — F1 captures the FN/FP trade-off directly.
  Logistic Regression achieves higher Recall, so it misses fewer >50K
  individuals than the Decision Tree.
""")

## Task 4: Interpretability Check

In [ ]:
# ── Logistic Regression: coefficient analysis ─────────────────────────────────

# Step names inside logistic_pipeline: 'preprocessor' and 'classifier'
lr_classifier = logistic_pipeline.named_steps['classifier']

# get_feature_names_out() on the fitted ColumnTransformer returns names for
# BOTH numeric columns and all OHE-expanded categorical columns.
feature_names = logistic_pipeline.named_steps['preprocessor'].get_feature_names_out()

coef_series = pd.Series(lr_classifier.coef_[0], index=feature_names)

top10_pos = coef_series.nlargest(10)
top10_neg = coef_series.nsmallest(10)

print('Top 10 POSITIVE coefficients  →  push model toward >50K prediction')
print(top10_pos.to_string())
print("""
  Interpretation (examples):
  • num__capital-gain      : Higher capital gains are a strong signal of wealth.
  • num__hours-per-week    : More hours worked correlates with higher income.
  • num__education-num     : More years of education increases earning potential.
  • cat__relationship_Husband: Married male head-of-household status is correlated
                               with higher income in this dataset.
  • cat__occupation_Exec-managerial / Prof-specialty: High-skill / managerial roles
                               command higher salaries.
""")

print('Top 10 NEGATIVE coefficients  →  push model toward <=50K prediction')
print(top10_neg.to_string())
print("""
  Interpretation (examples):
  • cat__relationship_Own-child / Not-in-family: Younger, dependent, or
                               single individuals tend to earn less.
  • cat__marital-status_Never-married: Younger / earlier-career stage.
  • cat__occupation_Other-service / Handlers-cleaners: Low-wage occupations.
  • num__capital-loss         : Negative, but note capital losses can be complex
                               signals — high loss may also indicate investments.
""")

In [ ]:
# ── Decision Tree: depth, train vs. test score, top splits ───────────────────
dt_classifier = tree_pipeline.named_steps['classifier']

tree_depth  = dt_classifier.get_depth()
train_score = tree_pipeline.score(X_train, y_train)
test_score  = tree_pipeline.score(X_test,  y_test)

print(f'Decision Tree depth   : {tree_depth}')
print(f'Train accuracy        : {train_score:.4f}')
print(f'Test  accuracy        : {test_score:.4f}')
print()

if train_score - test_score > 0.05:
    print('⚠️  Significant overfitting detected (train >> test).')
    print('   An unconstrained tree memorises training noise.')
    print('   Solution for Day 3: limit max_depth or use min_samples_leaf.')
else:
    print('✅  Train and test scores are close — little overfitting.')

# Export top 3 levels of the tree for readability
# We need the feature names the tree actually sees (post-preprocessing)
all_feature_names = tree_pipeline.named_steps['preprocessor'].get_feature_names_out()

print('\n--- Top 3 splits of the Decision Tree ---')
tree_rules = export_text(dt_classifier, feature_names=list(all_feature_names), max_depth=3)
print(tree_rules)

print("""
Comment on logic:
  The root split typically uses a highly discriminative feature such as
  'marital-status_Married-civ-spouse', 'capital-gain', or 'education-num'.
  This aligns with domain knowledge (and the logistic regression coefficients)
  confirming those are the most powerful predictors in this dataset.
  The tree's logic therefore appears sensible and interpretable.
""")

## Task 5: Write-Up & Model Selection for Day 3

### Model Selection

**Primary model for Day 3 — Logistic Regression.**
Across every metric (Accuracy, F1, ROC AUC, PR AUC), Logistic Regression outperforms the unconstrained Decision Tree on the hold-out test set. Crucially, the Decision Tree's training accuracy is near-perfect while its test accuracy drops sharply, revealing severe overfitting — the tree has grown deep enough to memorise the training data rather than generalise. Logistic Regression, by contrast, generalises well because L2 regularisation (lbfgs solver, default C=1.0) penalises overly complex weight vectors.

**Secondary experiment — constrained Decision Tree / Random Forest.**
Before abandoning tree-based models, we plan to restrict `max_depth` and tune `min_samples_leaf` to build a pruned tree that avoids memorisation. If that still underperforms, we will escalate to a Random Forest (ensemble of pruned trees) which exploits the tree's ability to model non-linear interactions while controlling variance through bagging.

### Preprocessing Notes for Day 3
- The saved `preprocessor.pkl` (below) will be re-used as-is to avoid rebuilding and re-fitting the transformer.
- Planned experiments: test `class_weight='balanced'` in Logistic Regression to improve Recall on the minority class; try `TargetEncoder` as an alternative to OHE for high-cardinality columns (e.g., occupation).

In [ ]:
# Save the fitted preprocessor for reuse on Day 3 (no data leakage risk
# because it was fit only on X_train)
joblib.dump(preprocessor, 'preprocessor.pkl')
print('preprocessor.pkl saved — ready for Day 3.')